In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../src")
import pandas as pd
from collections import Counter
from difflib import SequenceMatcher
from areas import candidates, summary_candidates, build_gazetteer, normalise, valid

pd.set_option("display.max_colwidth", 120)
df = pd.read_parquet("../data/processed/karachi_events_clean.parquet")

# Every candidate name, including one-off names
all_counts = Counter()
for location, city_raw, summary in zip(df["location"], df["city_raw"], df["summary_clean"]):
    names = set(candidates(location)) | set(summary_candidates(summary))
    if isinstance(city_raw, str):
        n = normalise(city_raw)
        if valid(n):
            names.add(n)
    all_counts.update(names)

gaz = build_gazetteer(df)
print("All candidate names:", len(all_counts), "| in the gazetteer (2+ events):", len(gaz))

All candidate names: 459 | in the gazetteer (2+ events): 97


In [2]:
# Pairs of similar names, most frequent first
names = list(all_counts)
pairs = []
for i, a in enumerate(names):
    for b in names[i + 1:]:
        if abs(len(a) - len(b)) <= 4:
            ratio = SequenceMatcher(None, a.lower(), b.lower()).ratio()
            if ratio >= 0.8:
                pairs.append((round(ratio, 2), a, all_counts[a], b, all_counts[b]))
pairs.sort(key=lambda p: -(p[2] + p[4]))
print(pd.DataFrame(pairs, columns=["similarity", "name_a", "events_a", "name_b", "events_b"]).head(60).to_string(index=False))

 similarity                name_a  events_a                name_b  events_b
       0.92                Orangi       146               Korangi        57
       0.80                Orangi       146             Chowrangi         1
       0.80       Gulshan-e-Iqbal        91       Gulshen e Iqbal         1
       0.80       Gulshan-e-Iqbal        91       Gulshan-e-Bihar         1
       0.82             Nazimabad        46              Azizabad         9
       0.83          Qasba Colony        28          Iqbal Colony         1
       0.80          Qasba Colony        28         Hassan Colony         1
       0.82           Sohrab Goth        21           Sochal Goth         1
       0.87         North Karachi        12     Northwest Karachi         1
       0.80         Baloch Colony         3          Bilal Colony         7
       0.83          Bihar Colony         2          Bilal Colony         7
       0.80            Pak Colony         6            PIB Colony         3
       0.82 

In [3]:
# Names that need a human decision: is "Gulshan" always Gulshan-e-Iqbal?
mask = df["location"].str.contains(r"\bGulshan\b(?![-\w])", na=False)
print(mask.sum(), "events with a standalone 'Gulshan'")
print(df.loc[mask, "location"].head(10).to_string())

# Different ways of writing Defence Housing Authority
print([(n, all_counts[n]) for n in all_counts if "defen" in n.lower() or n.lower() == "dha"])

8 events with a standalone 'Gulshan'
279     In block Z within the limits of Gulshan Maymar police station in Karachi.
443            The event occurred in the Taiser area of Gulshan e Iqbal, Karachi.
445        The event occurred in the PIB Colony area of Gulshan e Iqbal, Karachi.
454     The event occurred in the Gulistan e Johar area of Gulshan Town, Karachi.
836     The incident occurred in the Essa Nagri area of Gulshan Town in the city.
931                            Incident occurred in the Gulshan area of the city.
1138        The incident occurred in the Gulshan Hadeed neighborhood of the town.
1267                 The incident occurred in the Gulshan Iqbal area of the city.
[('Defense', 3), ('Defence Housing Authority', 12), ('Defense Housing', 1), ('Defence', 2), ('Defense Housing Authority', 4), ('Defense House Authority', 1)]


In [3]:
gaz = build_gazetteer(df)

pairs = Counter()
for loc in df["location"].dropna():
    found = list(dict.fromkeys(c for c in candidates(loc) if c in gaz))
    if len(found) >= 2:
        pairs[(found[0], found[1])] += 1

table = pd.DataFrame([(a, b, n) for (a, b), n in pairs.most_common()],
                     columns=["named_first", "then_named", "events"])
print(len(table), "different pairs")
print(table.to_string(index=False))

75 different pairs
            named_first                      then_named  events
                  Zaman                         Korangi       3
               Azizabad                         Gulberg       3
           Paposh Nagar                       Nazimabad       3
            Power House                   North Karachi       2
              Ram Swami                          Garden       2
           Qasba Colony                         Pirabad       2
            Safari Park                 Gulshan-e-Iqbal       2
             PIB Colony                 Gulshan-e-Iqbal       2
      Gulistan-e-Jauhar                 Gulshan-e-Iqbal       2
               Saudabad                           Malir       2
           Qasba Colony                            SITE       2
        Frontier Colony                          Orangi       2
          Ayesha Manzil                         Gulberg       2
                Pirabad                    Qasba Colony       2
              Manghop

In [2]:
from areas import extract_area
res = df.apply(lambda r: extract_area(r["location"], r["summary_clean"], r["city_raw"], gaz),
               axis=1, result_type="expand")
df["area"], df["area_method"] = res[0], res[1]
print("Gazetteer size:", len(gaz), "| coverage:", round(df["area"].notna().mean(), 2))
print(df["area_method"].value_counts(dropna=False))
print(df["area"].value_counts().head(25))

Gazetteer size: 97 | coverage: 0.74
area_method
location_pattern     1086
None                  425
summary_pattern        73
summary_gazetteer      55
city_field             25
Name: count, dtype: int64
area
Orangi                       169
Gulshan-e-Iqbal               98
North Nazimabad               65
Malir                         64
Korangi                       63
Lyari                         61
SITE                          54
Nazimabad                     52
Baldia                        46
Liaquatabad                   39
Gulistan-e-Jauhar             38
Gulberg                       34
Landhi                        34
Saddar                        33
Gadap                         22
Defence Housing Authority     22
Quaidabad                     18
New Karachi                   18
Clifton                       15
Jamshed                       14
North Karachi                 13
Surjani                       12
Garden                        11
Kharadar                      10

In [3]:
from areas import extract_area_detail
res = df.apply(lambda r: extract_area_detail(r["location"], r["summary_clean"], r["city_raw"], gaz),
               axis=1, result_type="expand")
df["area"], df["locality"], df["area_method"] = res[0], res[1], res[2]
print("Events with a more specific locality:", df["locality"].notna().sum())
for a in ["Gadap", "Orangi", "SITE", "Malir"]:
    print(a, df.loc[df["area"] == a, "locality"].value_counts(dropna=False).to_dict())

Events with a more specific locality: 225
Gadap {'Gulshan-e-Maymar': 10, nan: 7, 'Taiser': 3, 'Ahsanabad': 2}
Orangi {nan: 129, 'Manghopir': 22, 'Mominabad': 9, 'Frontier Colony': 7, 'Chisti Nagar': 2}
SITE {'Qasba Colony': 23, nan: 17, 'Pak Colony': 7, 'Metroville': 5, 'Islamia Colony': 2}
Malir {nan: 33, 'Sohrab Goth': 25, 'Saudabad': 3, 'Ibrahim Hyderi': 3}
